<a href="https://colab.research.google.com/github/EnomisLP/DiverseVul--IS-Project/blob/prashant/DiverseVul-%20IS%20Project%5Cvuln-detection%5Cnotebook%5Ccs2_exp4_lora.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



# CS2 EXP-4 — NeoBERT LoRA Fine-Tuning (Nested Rank Selection + Canonical Retrain)

Fine-tuning end-to-end del NeoBERT encoder tramite LoRA (Low-Rank Adaptation), con selezione nested del rank su 5 outer fold (ricerca inner a 3 fold), refit finale sul development completo e valutazione sul 20% outer holdout congelato — stesso split progetto-grouped riusato da CS1 EXP-0/1/2 e da CS2 EXP-3, per garantire confrontabilità.

A differenza di EXP-3 (probe lineare su encoder frozen), qui l'intero encoder NeoBERT viene aggiornato tramite adapter LoRA a basso rango: l'intera procedura nested (inner rank-search + outer refit + retrain canonico + holdout eval) è incapsulata in run_exp4_nested_pipeline (modulo case_study_2/exp4/exp4_nested_pipeline.py), con checkpoint automatico dopo ogni outer fold — se Colab si disconnette, ri-eseguire la stessa cella riprende da dove si era interrotto, senza ripartire da zero.


## 1. Runtime settings

In [ ]:
from pathlib import Path
REPO_URL = "https://github.com/EnomisLP/DiverseVul--IS-Project.git"
REPO_BRANCH = "prashant"
REPO_ROOT = Path("/content/DiverseVul--IS-Project")
PROJECT_DIR = REPO_ROOT / "vuln-detection"
SRC_DIR = PROJECT_DIR / "src"
DRIVE_ROOT = Path("/content/drive/MyDrive/IntelligentSystemProject/VulnerabilityDetectionData")
PROCESSED_DIR = DRIVE_ROOT / "processed"
MANIFEST_ROOT = DRIVE_ROOT / "manifests"
OUTPUT_ROOT = DRIVE_ROOT / "outputs"

# Stesso split project-grouped usato da CS1 EXP-0/1/2 and da CS2 EXP-3, per confrontabilita'.
SPLIT_ID = "cs1_project_holdout20_innercv_v1"

# NOTE: lo script exp4_nested_pipeline.py di default punta al parquet "abstracted"
# (rdiversevul_cs1_normalized_plus_abstracted_v1.parquet). Su richiesta, usiamo invece
# lo stesso normalized_v1.parquet di EXP-0/1/2/3 per rendere i risultati confrontabili.
NORMALIZED_PARQUET = PROCESSED_DIR / "rdiversevul_cs1_normalized_v1.parquet"
OUTER_MANIFEST_PATH = MANIFEST_ROOT / SPLIT_ID / "outer_holdout" / "cs1_outer_project_holdout_manifest.parquet"
INNER_MANIFEST_PATH = MANIFEST_ROOT / SPLIT_ID / "inner_cv" / "cs1_project_grouped_5fold_manifest.parquet"
EXP4_OUTPUT_DIR = OUTPUT_ROOT / "case_study_2" / "exp4_lora_nested_v1"
HF_CACHE_DIR = "/content/hf_cache"
RUN_PROFILE = False        # optional: single small LoRA fit, sanity check su tempo/VRAM
RUN_NESTED_LORA = True     # esegue l'intera pipeline nested (rank-search + refit + holdout eval)

# Ampliata verso il basso e verso l'alto rispetto al default dello script (8, 16),
# come controllo di validita' sui bordi della griglia (stesso principio applicato
# alla griglia di C in EXP-3).
RANK_GRID = (4, 8, 16, 32)
EPOCHS = 3

print("Settings loaded.")
print("Normalized parquet:", NORMALIZED_PARQUET)
print("Output dir:", EXP4_OUTPUT_DIR)
print("Rank grid:", RANK_GRID, "| epochs per fit:", EPOCHS)

# ATTENZIONE COSTO COMPUTAZIONALE:
# con 4 rank candidati, 5 outer fold e ricerca inner a 3 fold, la nested rank-search
# esegue 5 * 4 * 3 = 60 fine-tuning LoRA da EPOCHS epoche ciascuno, piu' 5 outer refit
# e 1 canonical retrain finale = 66 fit totali. Con il default (8, 16) sarebbero stati 36.
# Il checkpoint automatico e' dopo ogni OUTER FOLD (non dopo ogni rank), quindi in caso
# di disconnessione Colab a meta' della ricerca inner di un fold, quel fold riparte da capo.


## 2. Mount Google Drive and clone/refresh repository

In [ ]:
from google.colab import drive
import os
import subprocess
import sys

def run_command(command, cwd=None):
    print("$", " ".join(str(x) for x in command))
    subprocess.run(command, check=True, cwd=cwd)

drive.mount("/content/drive")

if not REPO_ROOT.exists():
    run_command([
        "git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, str(REPO_ROOT)
    ])
else:
    run_command(["git", "-C", str(REPO_ROOT), "fetch", "origin", REPO_BRANCH])
    run_command(["git", "-C", str(REPO_ROOT), "checkout", REPO_BRANCH])
    run_command(["git", "-C", str(REPO_ROOT), "pull", "--ff-only", "origin", REPO_BRANCH])

if not PROJECT_DIR.is_dir():
    raise FileNotFoundError(f"Missing project directory: {PROJECT_DIR}")
if not SRC_DIR.is_dir():
    raise FileNotFoundError(f"Missing source directory: {SRC_DIR}")

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("
Repository ready.")
run_command(["git", "-C", str(REPO_ROOT), "log", "-1", "--oneline"])


## 3. Verify GPU runtime (LoRA fine-tuning is GPU-only)

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "EXP-4 requires a GPU runtime (Runtime > Change runtime type > GPU). "
        "LoRA fine-tuning of NeoBERT is not supported on CPU in this pipeline."
    )

DEVICE = "cuda"
print("CUDA device:", torch.cuda.get_device_name(0))
print("bfloat16 supported:", torch.cuda.is_bf16_supported())

if not torch.cuda.is_bf16_supported():
    print(
        "WARNING: bfloat16 non supportato su questa GPU. train_lora_model usa "
        "torch.amp.autocast(dtype=torch.bfloat16) incondizionatamente: su GPU senza "
        "supporto bf16 nativo (es. T4) l'autocast puo' fallback silenziosamente o "
        "essere piu' lento. Preferire una GPU A100/L4 se disponibile."
    )


## 4. Install/import dependencies and verify committed files

In [ ]:
import subprocess
import sys

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers", "torch", "numpy", "pandas", "scikit-learn", "matplotlib",
    "pyarrow", "joblib", "peft",
], check=True)

print("Dependencies installed/verified.")


In [ ]:
# NOTE: si assume che i due file forniti siano committati nella repo con questi path,
# seguendo la stessa convenzione di case_study_2/exp3/exp3_linear_probe.py:
# - case_study_2/exp4/exp4_lora.py            -> train_lora_model
# - case_study_2/exp4/exp4_nested_pipeline.py -> run_exp4_nested_pipeline
# Se i path reali sono diversi, aggiornare questa lista e le import sotto.
required_repo_files = [
    SRC_DIR / "case_study_1" / "split_manifest.py",
    SRC_DIR / "case_study_1" / "evaluation.py",
    SRC_DIR / "case_study_1" / "confidence_intervals.py",
    SRC_DIR / "case_study_2" / "data_loader.py",
    SRC_DIR / "case_study_2" / "models.py",
    SRC_DIR / "case_study_2" / "exp4" / "exp4_lora.py",
    SRC_DIR / "case_study_2" / "exp4" / "exp4_nested_pipeline.py",
]

missing = [str(p) for p in required_repo_files if not p.is_file()]
if missing:
    raise FileNotFoundError("Missing required committed files:
" + "
".join(missing))

from case_study_1 import split_manifest
from case_study_1 import confidence_intervals
from case_study_2.data_loader import create_dataloader
from case_study_2.models import configure_huggingface_cache, load_neobert_tokenizer, DEFAULT_NEOBERT_TOKENIZER
from case_study_2.exp4.exp4_lora import train_lora_model
from case_study_2.exp4.exp4_nested_pipeline import run_exp4_nested_pipeline

print("Imported project modules successfully.")


## 5. Load dataset and frozen manifests

In [ ]:
import pandas as pd

if not NORMALIZED_PARQUET.is_file():
    raise FileNotFoundError(f"Missing normalized dataset: {NORMALIZED_PARQUET}")
if not OUTER_MANIFEST_PATH.is_file():
    raise FileNotFoundError(f"Missing outer manifest: {OUTER_MANIFEST_PATH}")
if not INNER_MANIFEST_PATH.is_file():
    raise FileNotFoundError(f"Missing inner manifest: {INNER_MANIFEST_PATH}")

full_df = pd.read_parquet(NORMALIZED_PARQUET)
outer_manifest_df = pd.read_parquet(OUTER_MANIFEST_PATH)
inner_manifest_df = split_manifest.load_manifest(
    INNER_MANIFEST_PATH,
    config=split_manifest.SplitConfig(n_splits=5, random_state=42, shuffle=True),
)

required_columns = {"source_row_id", "normalized_code", "label", "project"}
missing_columns = required_columns.difference(full_df.columns)
if missing_columns:
    raise KeyError(f"Dataset missing required columns: {sorted(missing_columns)}")

print("Full dataset:", full_df.shape)
print("Outer manifest:", outer_manifest_df.shape)
print("Inner manifest:", inner_manifest_df.shape)
print(outer_manifest_df["partition"].value_counts(dropna=False))


## 6. Strict manifest and development-only validation

In [ ]:
for name, frame in [("full_df", full_df), ("outer_manifest_df", outer_manifest_df), ("inner_manifest_df", inner_manifest_df)]:
    if frame["source_row_id"].duplicated().any():
        raise RuntimeError(f"{name} contains duplicate source_row_id values.")

full_indexed = full_df.set_index("source_row_id", drop=False)
dev_ids = set(outer_manifest_df.loc[outer_manifest_df["partition"] == "development", "source_row_id"].tolist())
holdout_ids = set(outer_manifest_df.loc[outer_manifest_df["partition"] == "outer_holdout", "source_row_id"].tolist())
inner_ids = set(inner_manifest_df["source_row_id"].tolist())

if dev_ids.intersection(holdout_ids):
    raise RuntimeError("Development and outer holdout ID overlap detected.")

if inner_ids != dev_ids:
    raise RuntimeError(
        f"Inner CV manifest IDs must exactly equal development IDs. "
        f"Missing={len(dev_ids - inner_ids)}, extra={len(inner_ids - dev_ids)}"
    )

dev_projects = set(outer_manifest_df.loc[outer_manifest_df["partition"] == "development", "project"].astype(str))
holdout_projects = set(outer_manifest_df.loc[outer_manifest_df["partition"] == "outer_holdout", "project"].astype(str))

if dev_projects.intersection(holdout_projects):
    raise RuntimeError("Development/holdout project overlap detected.")

development_frame = full_indexed.loc[list(dev_ids)].copy().reset_index(drop=True)
holdout_frame = full_indexed.loc[list(holdout_ids)].copy().reset_index(drop=True)

for frame in (development_frame, holdout_frame):
    frame["normalized_code"] = frame["normalized_code"].fillna("").astype(str)
    empty_mask = frame["normalized_code"].str.strip().eq("")
    if empty_mask.any():
        frame.loc[empty_mask, "normalized_code"] = "EMPTY_CODE_SAMPLE"

print("Development rows:", len(development_frame), "| projects:", development_frame["project"].nunique())
print("Outer holdout rows:", len(holdout_frame), "| projects:", holdout_frame["project"].nunique())
print("Leakage check passed: development/outer holdout partitions and projects are disjoint.")


## 7. Configure EXP-4 LoRA nested pipeline

In [ ]:
from case_study_2.models import DEFAULT_NEOBERT_MODEL

print("Base model:", DEFAULT_NEOBERT_MODEL, "| Tokenizer:", DEFAULT_NEOBERT_TOKENIZER)
print("Rank grid:", RANK_GRID)
print("Epochs per fit:", EPOCHS)
print("Output directory:", EXP4_OUTPUT_DIR)
print("HF cache dir:", HF_CACHE_DIR)


## 8. Optional profile run — single LoRA fit sanity check (speed & VRAM)

Prima di lanciare l'intera pipeline nested (potenzialmente decine di fit LoRA), eseguiamo un singolo fit veloce su un piccolo sottoinsieme del development set, con epochs=1 e il rank più piccolo della griglia, solo per verificare tempo per epoca e VRAM di picco su questa GPU/runtime — non è un tentativo di ottimizzazione, è un controllo di validità/fattibilità prima di un job lungo.


In [ ]:
if RUN_PROFILE:
    profile_sample = development_frame.sample(n=min(400, len(development_frame)), random_state=42)
    profile_train_df = profile_sample.iloc[: int(0.8 * len(profile_sample))]
    profile_val_df = profile_sample.iloc[int(0.8 * len(profile_sample)):]
    tokenizer = load_neobert_tokenizer(DEFAULT_NEOBERT_TOKENIZER, hf_cache_dir=HF_CACHE_DIR)

    profile_scores, profile_model = train_lora_model(
        profile_train_df, profile_val_df, tokenizer,
        rank=RANK_GRID[0], epochs=1, device=DEVICE, hf_cache_dir=HF_CACHE_DIR,
    )

    from sklearn.metrics import average_precision_score
    profile_prauc = average_precision_score(profile_val_df["label"].values, profile_scores)
    print(f"Profile PR-AUC (small subsample, not meaningful on its own): {profile_prauc:.4f}")

    del profile_model
    import gc
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
else:
    print("RUN_PROFILE=False; skipping profile run.")


## 9. Official nested LoRA pipeline (rank selection + outer refit + canonical retrain + holdout eval)

run_exp4_nested_pipeline incapsula l'intera procedura: per ciascuno dei 5 outer fold del development set, seleziona il rank ottimo tramite ricerca inner a 3 fold su RANK_GRID, fa il refit sull'outer-train con il rank selezionato e valuta sull'outer-val (OOF pooled); poi determina il rank globale come moda dei rank selezionati nei 5 fold, fa il retrain canonico su tutto il development set con quel rank, e valuta sul 20% outer holdout congelato. Tutto in un'unica chiamata, con checkpoint automatico dopo ogni outer fold (in EXP4_OUTPUT_DIR/exp4_nested_checkpoint.joblib): se il runtime Colab si disconnette, ri-eseguire questa stessa cella riprende automaticamente dai fold già completati.


In [ ]:
if RUN_NESTED_LORA:
    run_exp4_nested_pipeline(
        abstracted_parquet_path=str(NORMALIZED_PARQUET),
        inner_manifest_path=str(INNER_MANIFEST_PATH),
        outer_manifest_path=str(OUTER_MANIFEST_PATH),
        output_dir_path=str(EXP4_OUTPUT_DIR),
        hf_cache_dir=HF_CACHE_DIR,
        rank_grid=RANK_GRID,
        epochs=EPOCHS,
    )
    print("
Official EXP-4 nested LoRA pipeline complete.")
else:
    print("RUN_NESTED_LORA=False; skipping official nested LoRA pipeline.")


## 10. Load pipeline outputs and display nested rank-selection summary

In [ ]:
oof_path = EXP4_OUTPUT_DIR / "exp4_oof_predictions.csv"
ranks_path = EXP4_OUTPUT_DIR / "exp4_selected_rank_per_outer_fold.csv"
holdout_pred_path = EXP4_OUTPUT_DIR / "exp4_holdout_predictions.csv"

if not oof_path.is_file():
    raise FileNotFoundError(
        f"Missing {oof_path}. Run Section 9 first (RUN_NESTED_LORA=True)."
    )

exp4_oof_df = pd.read_csv(oof_path)
exp4_selected_ranks_df = pd.read_csv(ranks_path)

print("Selected rank by outer fold:")
display(exp4_selected_ranks_df)
print("
Rank frequency across outer folds:")
display(exp4_selected_ranks_df["selected_rank"].value_counts())

global_selected_rank = int(exp4_selected_ranks_df["selected_rank"].mode()[0])
print("
Global canonical rank (mode across outer folds):", global_selected_rank)


## 11. Development nested pooled OOF metrics

In [ ]:
from sklearn.metrics import average_precision_score, roc_auc_score

pooled_pr_auc = average_precision_score(exp4_oof_df["label"], exp4_oof_df["y_score"])
pooled_roc_auc = roc_auc_score(exp4_oof_df["label"], exp4_oof_df["y_score"])

print("Pooled nested OOF metrics (EXP-4 LoRA, development):")
display(pd.DataFrame([
    {"metric": "average_precision_pr_auc", "value": pooled_pr_auc},
    {"metric": "roc_auc", "value": pooled_roc_auc},
]))

print("
Per-outer-fold PR-AUC:")
fold_pr_auc = (
    exp4_oof_df.groupby("fold")
    .apply(lambda g: average_precision_score(g["label"], g["y_score"]))
    .rename("pr_auc")
    .reset_index()
)
display(fold_pr_auc)


## 11b. Confidence interval on development-CV pooled PR-AUC

In [ ]:
exp4_devcv_ci = confidence_intervals.bootstrap_metric_ci(
    exp4_oof_df,
    metric="average_precision_pr_auc",
    n_bootstrap=1000,
    random_state=42,
)
print(confidence_intervals.format_ci_report(exp4_devcv_ci))


## 12. Compare against previous development-only reference results

In [ ]:
reference_rows = [
    {"experiment": "EXP-0 fixed normalized_code", "scope": "development pooled OOF", "pr_auc": 0.125205},
    {"experiment": "EXP-0 nested-alpha normalized_code", "scope": "development nested pooled OOF", "pr_auc": 0.141971},
    {"experiment": "EXP-2 MLP fixed representation", "scope": "development pooled OOF", "pr_auc": 0.138157},
    {
        "experiment": f"EXP-4 NeoBERT LoRA fine-tune (nested rank, rank_grid={RANK_GRID})",
        "scope": "development nested pooled OOF",
        "pr_auc": float(pooled_pr_auc),
    },
]

# Se hai gia' in mano il numero ufficiale di EXP-3 (dal notebook EXP-3), aggiungilo qui
# a mano per un confronto diretto, es.:
# reference_rows.append({"experiment": "EXP-3 NeoBERT linear probe (nested C, ufficiale)",
# "scope": "development nested pooled OOF", "pr_auc": <valore>})

comparison_df = pd.DataFrame(reference_rows).sort_values("pr_auc", ascending=False).reset_index(drop=True)
display(comparison_df)


## 13. Final evaluation: score frozen 20% outer holdout partition

In [ ]:
if not holdout_pred_path.is_file():
    raise FileNotFoundError(
        f"Missing {holdout_pred_path}. Run Section 9 first (RUN_NESTED_LORA=True)."
    )

exp4_holdout_df = pd.read_csv(holdout_pred_path)
DECISION_THRESHOLD = 0.50
exp4_holdout_df["y_pred"] = (exp4_holdout_df["y_score"] >= DECISION_THRESHOLD).astype(int)

holdout_pr_auc = average_precision_score(exp4_holdout_df["label"], exp4_holdout_df["y_score"])
holdout_roc_auc = roc_auc_score(exp4_holdout_df["label"], exp4_holdout_df["y_score"])

print("Outer-holdout metrics (EXP-4 LoRA, canonical rank =", global_selected_rank, "):")
display(pd.DataFrame([
    {"metric": "average_precision_pr_auc", "value": holdout_pr_auc},
    {"metric": "roc_auc", "value": holdout_roc_auc},
]))


## 13b. Confidence interval on outer-holdout PR-AUC

In [ ]:
exp4_holdout_ci = confidence_intervals.bootstrap_metric_ci(
    exp4_holdout_df,
    metric="average_precision_pr_auc",
    n_bootstrap=1000,
    random_state=42,
)
print(confidence_intervals.format_ci_report(exp4_holdout_ci))


## 14. Interpretability note

A differenza di EXP-3 (probe lineare, dove i pesi coef_ sono direttamente interpretabili per dimensione dell'embedding), qui l'intero encoder NeoBERT è stato aggiornato tramite adapter LoRA end-to-end: non esiste un vettore di pesi lineare unico da ispezionare. Un'analisi di interpretabilità comparabile richiederebbe tecniche diverse (es. attention rollout, gradient-based saliency sui token di input, o probing degli adapter LoRA stessi) che sono fuori scope per questo notebook. L'adapter canonico finale è salvato in EXP4_OUTPUT_DIR/final_canonical_lora_model per eventuali analisi future.


## 15. Error analysis: isolate holdout false positives & false negatives

In [ ]:
holdout_res = holdout_frame.merge(
    exp4_holdout_df[["source_row_id", "y_score", "y_pred"]],
    on="source_row_id",
    how="inner",
)

if len(holdout_res) != len(holdout_frame):
    raise RuntimeError(
        "Merge tra holdout_frame e le predizioni EXP-4 ha perso/duplicato righe: "
        f"holdout_frame={len(holdout_frame)}, merged={len(holdout_res)}"
    )

false_positives = holdout_res[(holdout_res["label"] == 0) & (holdout_res["y_pred"] == 1)]
false_negatives = holdout_res[(holdout_res["label"] == 1) & (holdout_res["y_pred"] == 0)]

print(f"Extracted {len(false_positives)} False Positives and {len(false_negatives)} False Negatives.")

false_positives[["source_row_id", "project", "y_score", "normalized_code"]].sample(
    n=min(5, len(false_positives)), random_state=42
).to_csv(EXP4_OUTPUT_DIR / "sample_false_positives.csv", index=False)

false_negatives[["source_row_id", "project", "y_score", "normalized_code"]].sample(
    n=min(5, len(false_negatives)), random_state=42
).to_csv(EXP4_OUTPUT_DIR / "sample_false_negatives.csv", index=False)
